# pipe_nuevo — 07 bis: entrenamiento final CON fallback a naive por unidad

Variante de `07_Entrenamiento_final.ipynb` con una regla extra: cuando
`entrenar_por_cluster=True` en `06_Optuna.ipynb`, algunos clusters (los mas
chicos o mas ruidosos) pueden terminar con un LightGBM que NO le gana al
baseline naive en validacion -- entrenar un modelo complejo ahi solo suma
varianza al submit final. `06_Optuna` ya guarda `wape_val` Y `wape_naive_val`
en cada `resultado.json`, asi que la comparacion es directa, sin recalcular
nada: **por cada unidad (cluster o el pooled completo), si el LightGBM no
supera al naive en val, se usa naive (repetir `tn0`, la ultima venta
observada) para esa unidad; si lo supera, se entrena y se usa LightGBM.**

Mismo esqueleto que `07_Entrenamiento_final.ipynb` (elige experimento, carga,
join de cluster, reconstruccion a toneladas, submit a Kaggle) -- la unica
diferencia es `entrenar_y_predecir_unidad()` y el registro final, que ahora
deja anotada la regla usada por cada unidad.


In [ ]:
import gc, json, os, shutil, subprocess, time
from pathlib import Path

import numpy as np
import polars as pl
import pandas as pd
import lightgbm as lgb


def resolver_bucket() -> Path:
    env = os.environ.get("LABO3_BUCKET")
    if env:
        p = Path(env).expanduser().resolve()
        p.mkdir(parents=True, exist_ok=True)
        return p
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1", "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError(
        "No encontre el bucket. Defini LABO3_BUCKET, ej: "
        "os.environ['LABO3_BUCKET'] = '/home/usuario/labo3-bucket'"
    )


BUCKET   = resolver_bucket()
RUTA_FE  = BUCKET / "datasets_fe"
RUTA_EXP = BUCKET / "exp_pipe_nuevo"
RUTA_RAW = BUCKET / "datasets"

print(f"BUCKET: {BUCKET}")


def desplazar_meses(p: int, k: int) -> int:
    m = (p // 100) * 12 + (p % 100) - 1 + k
    return (m // 12) * 100 + (m % 12) + 1


### Elegir el experimento

Un experimento es o bien una carpeta con `resultado.json` (pooled), o una carpeta con `manifest.json` + una subcarpeta `cluster{K}/resultado.json` por cluster (por-cluster). Se comparan los dos tipos por `wape_test` (ponderado por tamaño de cluster en el segundo caso).


In [ ]:
def inventario_experimentos():
    entradas = []
    for d in sorted(RUTA_EXP.iterdir()):
        if not d.is_dir():
            continue
        f_manifest = d / 'manifest.json'
        f_res = d / 'resultado.json'
        if f_manifest.exists():
            m = json.load(open(f_manifest, encoding='utf-8'))
            entradas.append({'nombre': d.name, 'dir': d, 'modo': 'porCluster',
                             'wape_test': m.get('wape_test_ponderado')})
        elif f_res.exists():
            r = json.load(open(f_res, encoding='utf-8'))
            entradas.append({'nombre': d.name, 'dir': d, 'modo': 'pooled',
                             'wape_test': r.get('wape_test')})
    return entradas


DISPONIBLES = inventario_experimentos()
print(f"Experimentos disponibles en {RUTA_EXP.name}/:")
for e in DISPONIBLES:
    wt = e['wape_test']
    print(f"   [{e['modo']:9s}] wape_test={wt:.4f}   {e['nombre']}" if wt is not None
          else f"   [{e['modo']:9s}] wape_test=???     {e['nombre']}")
if not DISPONIBLES:
    raise RuntimeError(f"No hay experimentos en {RUTA_EXP}. Corre 06_Optuna.")


In [ ]:
PARAM = {
    # Nombre exacto de carpeta (columna 'nombre' de arriba). None = mejor wape_test.
    'experimento': None,

    # ── MUESTREO DE CLIENTES ─────────────────────────────────────────────
    # A diferencia de 06_, aca no hay val/test que proteger: se entrena con
    # TODOS los meses supervisados. El limite sigue siendo memoria.
    'muestreo_clientes': None,

    # Semillas del ensemble: un modelo por semilla, se promedian las predicciones.
    'semillas_ensemble': [102191],

    # Mes objetivo de la entrega. El pipe predice a t+horizonte.
    'periodo_objetivo': None,   # None = el ultimo periodo_objetivo disponible

    # Kaggle
    'kaggle_competition': 'labo-iii-2026-rosario',
    'submit': True,
    'mensaje_submit': None,

    'clip_min': 0.0,

    # ── FALLBACK A NAIVE POR UNIDAD (lo nuevo de este notebook) ──────────
    # True -> por cada unidad (cluster, o el pooled completo si no hay
    #         cluster), si wape_naive_val < wape_val (el LightGBM de 06_
    #         NO le gano al naive en validacion), esa unidad se predice
    #         repitiendo tn0 en vez de entrenar/usar el ensemble LightGBM.
    #         False -> se comporta igual que 07_Entrenamiento_final.ipynb
    #         (siempre LightGBM), para poder comparar los dos submits.
    'permitir_fallback_naive': True,
}

if PARAM['experimento'] is None:
    _con_wape = [e for e in DISPONIBLES if e['wape_test'] is not None]
    ELEGIDO = min(_con_wape or DISPONIBLES, key=lambda e: e['wape_test'] if e['wape_test'] is not None else float('inf'))
    print("Sin experimento indicado -> se elige el de mejor wape_test")
else:
    ELEGIDO = next((e for e in DISPONIBLES if e['nombre'] == PARAM['experimento']), None)
    if ELEGIDO is None:
        raise FileNotFoundError(f"No existe {PARAM['experimento']!r}.\n"
                                f"Disponibles: {[e['nombre'] for e in DISPONIBLES]}")

DIR_EXP = ELEGIDO['dir']
MODO = ELEGIDO['modo']
print(f"\nExperimento elegido: {ELEGIDO['nombre']}")
print(f"Modo: {MODO}   wape_test: {ELEGIDO['wape_test']}")


### Cargar la config (uno o varios `resultado.json` segun el modo)


In [ ]:
if MODO == 'porCluster':
    manifest = json.load(open(DIR_EXP / 'manifest.json', encoding='utf-8'))
    RESULTADOS = [json.load(open(p, encoding='utf-8')) for p in manifest['resultados']]
    ARCHIVO_CLUSTERS = manifest['archivo_clusters']
    COL_CLUSTER = manifest['col_cluster']
    print(f"Modo por-cluster: {len(RESULTADOS)} clusters -> {manifest['clusters']}")
else:
    RESULTADOS = [json.load(open(DIR_EXP / 'resultado.json', encoding='utf-8'))]
    ARCHIVO_CLUSTERS = RESULTADOS[0].get('archivo_clusters')
    COL_CLUSTER = RESULTADOS[0].get('col_cluster')

# Config compartida por todos los sub-experimentos (aca son todos consistentes
# entre si porque 06_Optuna corre el mismo PARAM base en todo el loop de clusters,
# pero se valida igual: un manifest.json tocado a mano podria no cumplirlo).
CFG0 = RESULTADOS[0]
DATASET_FE  = CFG0['dataset_fe']
TARGET      = CFG0['target']
TARGET_KIND = CFG0['target_kind']
METODO      = CFG0['metodo_normalizacion']
H           = CFG0['horizonte']

for _campo in ('dataset_fe', 'target', 'target_kind', 'metodo_normalizacion', 'horizonte'):
    _valores = {r[_campo] for r in RESULTADOS}
    if len(_valores) > 1:
        raise ValueError(f"Los sub-experimentos de cluster no coinciden en {_campo!r}: "
                         f"{_valores}. El manifest.json esta corrupto o mezcla corridas distintas.")

print(f"Dataset FE  : {DATASET_FE}")
print(f"Target      : {TARGET}  (kind={TARGET_KIND}, norm={METODO}, horizonte={H})")
print(f"Cluster     : {ARCHIVO_CLUSTERS or '(no se uso)'}")
for r in RESULTADOS:
    etiqueta = f"cluster {r['cluster']}" if r.get('cluster') is not None else "pooled"
    print(f"  [{etiqueta}] wape_val={r.get('wape_val'):.4f}  wape_test={r.get('wape_test'):.4f}  "
          f"n_estimators_final={r.get('n_estimators_final')}  n_features={r['n_features']}")


### Regla naive-vs-LightGBM por unidad

Se decide ACA, antes de entrenar nada: para cada unidad, si
`wape_naive_val < wape_val` (el naive le gano al LightGBM de `06_Optuna` en
validacion), esa unidad usa naive -- no hace falta ni entrenarla. Comparacion
directa sobre numeros que `06_` ya calculo y guardo.


In [ ]:
REGLA_POR_UNIDAD = {}
for r in RESULTADOS:
    cl = r.get('cluster')
    wv, wnv = r.get('wape_val'), r.get('wape_naive_val')
    if not PARAM['permitir_fallback_naive'] or wv is None or wnv is None:
        regla = 'lightgbm'
    else:
        regla = 'lightgbm' if wv < wnv else 'naive'
    REGLA_POR_UNIDAD[cl] = regla
    etiqueta = f"cluster {cl}" if cl is not None else "pooled"
    marca = "" if regla == 'lightgbm' else "  <- LightGBM NO le gana al naive, se usa naive"
    print(f"  [{etiqueta}] wape_val={wv:.4f}  wape_naive_val={wnv:.4f}  -> {regla}{marca}")

_n_naive = sum(1 for v in REGLA_POR_UNIDAD.values() if v == 'naive')
if _n_naive:
    print(f"\n{_n_naive} de {len(REGLA_POR_UNIDAD)} unidad(es) van a usar naive en vez de LightGBM.")
else:
    print("\nTodas las unidades le ganan al naive en val -> se comporta igual que 07_Entrenamiento_final.ipynb.")


### Carga: mismo tratamiento de memoria que `06_Optuna.ipynb`

Aca se entrena con TODO lo supervisado: los meses que `06_` había reservado
para val/test ya cumplieron su función. La inferencia (target nulo en los
últimos `H` meses) tampoco se muestrea nunca -- de ahí sale la entrega y no
puede faltar ningún producto.


In [ ]:
t0 = time.time()

CTX_F64 = {'B0', 'B1', 'tn0_norm', 'tn0', 'clase_tn', 'clase_tn_norm', 'clase_tn_delta'}
path_in = RUTA_FE / DATASET_FE
if not path_in.exists():
    raise FileNotFoundError(f"No existe {path_in}. Corre 01_Preprocesamiento -> 02_FE -> 03_Escalado.")

lf = pl.scan_parquet(path_in)
_schema = lf.collect_schema()
COLS_ALL = list(_schema.keys())
_f64 = [c for c, t in _schema.items() if t == pl.Float64]
_a_f32 = [c for c in _f64 if c not in CTX_F64]
lf = lf.with_columns([pl.col(c).cast(pl.Float32) for c in _a_f32])

periodos = sorted(lf.select('periodo').unique().collect()['periodo'].to_list())
print(f"Dataset: {len(COLS_ALL)} columnas   periodos {periodos[0]} -> {periodos[-1]}")

MESES_INFER = periodos[-H:]
_N_CLI = PARAM.get('muestreo_clientes')
if _N_CLI and 'customer_id' in COLS_ALL:
    _cli_ok = pl.col('customer_id').hash(seed=CFG0.get('semilla', 102191)) % int(_N_CLI) == 0
else:
    _cli_ok = pl.lit(True)

df_infer = lf.filter(pl.col(TARGET).is_null() & pl.col('periodo').is_in(MESES_INFER)).collect()
df_sup = lf.filter(pl.col(TARGET).is_not_null() & _cli_ok).collect()

_sup_disponibles = lf.select(pl.col(TARGET).is_not_null().sum()).collect().item()
meses_sup = sorted(df_sup['periodo'].unique().to_list())
print(f"\nSupervisado : {df_sup.height:,} de {_sup_disponibles:,} filas "
      f"({100*df_sup.height/_sup_disponibles:.0f}%)"
      + (f"   [1 de cada {_N_CLI} clientes]" if _N_CLI else "   [todos los clientes]"))
print(f"              meses {meses_sup[0]} -> {meses_sup[-1]}")
print(f"Inferencia  : {df_infer.height:,} filas   meses {sorted(df_infer['periodo'].unique().to_list())}")
print(f"[{time.time()-t0:.0f}s]")

if df_infer.height == 0:
    raise RuntimeError("No hay filas de inferencia: sin ellas no se puede armar la entrega.")


### Cluster DTW (si el experimento lo uso)


In [ ]:
if ARCHIVO_CLUSTERS:
    path_cl = RUTA_FE / ARCHIVO_CLUSTERS
    if not path_cl.exists():
        raise FileNotFoundError(f"No existe {path_cl} (el que uso 06_Optuna para este experimento).")
    clu = pl.read_parquet(path_cl).select(["product_id", "customer_id", COL_CLUSTER]) \
             .rename({COL_CLUSTER: "cluster"})
    df_sup = (df_sup.join(clu, on=["product_id", "customer_id"], how="left")
                    .with_columns(pl.col("cluster").fill_null(-1)))
    df_infer = (df_infer.join(clu, on=["product_id", "customer_id"], how="left")
                        .with_columns(pl.col("cluster").fill_null(-1)))
    print(f"cluster pegado desde {path_cl.name}")
    print(df_sup.group_by("cluster").agg(pl.len().alias("n_filas")).sort("cluster"))


### A pandas + liberar polars


In [ ]:
IDS = [c for c in ['product_id', 'customer_id'] if c in COLS_ALL]
COLS_CTX = [c for c in ['B0', 'B1', 'tn0_norm', 'clase_tn'] + IDS
           + (['cluster'] if ARCHIVO_CLUSTERS else []) if c in COLS_ALL or c == 'cluster']

TODAS_FEATURES = sorted({f for r in RESULTADOS for f in r['features']})
faltan = [c for c in TODAS_FEATURES if c not in COLS_ALL]
if faltan:
    raise ValueError(f"El dataset no tiene {len(faltan)} features del experimento: {faltan[:10]}")

TODAS_CAT = sorted({c for r in RESULTADOS for c in r['cat_features']})
_cols = sorted(set(TODAS_FEATURES + COLS_CTX + ['periodo'] + [TARGET]))
_cats = [c for c in TODAS_CAT if c in _cols]

train_pd = (df_sup.select(_cols)
                  .with_columns([pl.col(c).cast(pl.Categorical) for c in _cats])
                  .to_pandas())
infer_pd = (df_infer.select([c for c in _cols if c in df_infer.columns])
                    .with_columns([pl.col(c).cast(pl.Categorical)
                                   for c in _cats if c in df_infer.columns])
                    .to_pandas())
for c in TODAS_CAT:
    train_pd[c] = train_pd[c].astype('category')
    if c in infer_pd.columns:
        infer_pd[c] = infer_pd[c].astype('category').cat.set_categories(train_pd[c].cat.categories)

print(f"train: {train_pd.shape}   inferencia: {infer_pd.shape}")

for _v in ('df_sup', 'df_infer'):
    globals().pop(_v, None)
gc.collect()
print(f"polars liberado; train_pd: {train_pd.memory_usage(deep=True).sum()/1e9:.2f} GB   "
      f"infer_pd: {infer_pd.memory_usage(deep=True).sum()/1e9:.2f} GB")


### Reconstruccion a toneladas (compartida por todas las unidades)


In [ ]:
def reconstruir_nivel(pred, ctx) -> np.ndarray:
    pred = np.asarray(pred, dtype=np.float64)
    if TARGET_KIND == 'delta':
        pred = pred + ctx['tn0_norm'].to_numpy(dtype=np.float64)
    B0 = ctx['B0'].to_numpy(dtype=np.float64)
    B1 = ctx['B1'].to_numpy(dtype=np.float64)
    B1s = np.where((B1 == 0) | ~np.isfinite(B1), 1.0, B1)
    return pred * B1s + B0


# Chequeo de sanidad: reconstruir el TARGET REAL sobre train debe devolver clase_tn.
if 'clase_tn' in train_pd.columns:
    _m = train_pd.head(min(20_000, len(train_pd)))
    _rec = reconstruir_nivel(_m[TARGET].to_numpy(), _m)
    _err = float(np.nanmax(np.abs(_rec - _m['clase_tn'].to_numpy())))
    print(f"Round-trip de reconstruccion: error maximo = {_err:.10f}")
    if _err > 1e-6:
        raise RuntimeError(f"La reconstruccion no cierra (error {_err}). Revisa METODO={METODO!r}.")
    print("Reconstruccion validada.")
    del _m


### Cuanto va a tardar esto

Se calibra con pocos arboles sobre la unidad mas pesada (mas filas x arboles) y se extrapola a TODO el trabajo (todas las unidades x todas las semillas del ensemble). El tiempo de LightGBM crece casi lineal en n_estimators, asi que la cuenta es confiable.


In [ ]:
def _fmt(seg):
    if seg < 90: return f"{seg:.0f} s"
    if seg < 5400: return f"{seg/60:.1f} min"
    return f"{seg/3600:.1f} h"


UNIDADES = []
for r in RESULTADOS:
    cl = r.get('cluster')
    if REGLA_POR_UNIDAD[cl] == 'naive':
        continue   # no se entrena, no entra al costo estimado
    sub = train_pd[train_pd['cluster'] == cl] if cl is not None else train_pd
    UNIDADES.append({'resultado': r, 'n_filas': len(sub), 'n_estimators': r['n_estimators_final'],
                     'costo': len(sub) * r['n_estimators_final']})

if not UNIDADES:
    print("Todas las unidades usan naive -> no hay nada que entrenar ni que calibrar.")
else:
    _peor = max(UNIDADES, key=lambda u: u['costo'])
    _r, _n_filas = _peor['resultado'], _peor['n_filas']
    N_CALIB = 40
    _p = dict(_r['hiperparametros'])
    _p.update({'objective': _r['objective_lgbm'], 'metric': 'mae', 'verbosity': -1,
              'boosting_type': 'gbdt', 'n_jobs': -1, 'subsample_freq': 1,
              'n_estimators': N_CALIB, 'seed': PARAM['semillas_ensemble'][0]})
    _sub = (train_pd[train_pd['cluster'] == _r.get('cluster')] if _r.get('cluster') is not None else train_pd)

    _t0 = time.time()
    _m = lgb.LGBMRegressor(**_p)
    _m.fit(_sub[_r['features']], _sub[TARGET], categorical_feature=_r['cat_features'])
    _t_calib = time.time() - _t0
    del _m, _sub
    gc.collect()

    _seg_por_fila_arbol = _t_calib / (_n_filas * N_CALIB)
    _total = sum(u['costo'] for u in UNIDADES) * len(PARAM['semillas_ensemble']) * _seg_por_fila_arbol

    print(f"Unidades a entrenar con LightGBM: {len(UNIDADES)} de {len(RESULTADOS)}   "
          f"semillas: {len(PARAM['semillas_ensemble'])}")
    print(f"Calibracion con {N_CALIB} arboles sobre la unidad mas pesada: {_t_calib:.1f} s")
    print(f"\nESTIMADO total: {_fmt(_total)}")
    if _total > 3600:
        print("Mas de una hora. Si estas en una maquina spot, considera bajar "
              "'semillas_ensemble' a una sola o subir 'muestreo_clientes'.")


### Entrenar el ensemble y predecir

`entrenar_y_predecir_unidad()` es la unidad de trabajo: un cluster (o el
pooled completo) con SUS PROPIOS hiperparametros/`n_estimators_final`. En modo
por-cluster se llama una vez por cluster y se concatenan las predicciones
antes de agregar por producto -- el resto del notebook no distingue el modo.


In [ ]:
def predecir_naive(infer_sub, etiqueta):
    """Repite tn0 (la ultima venta observada) -- el baseline con el que se
    comparo en 06_Optuna. Sin entrenar nada."""
    if len(infer_sub) == 0:
        print(f"  [{etiqueta}] (naive) sin filas de inferencia")
        return pd.DataFrame(columns=IDS + ['periodo', 'periodo_objetivo', 'tn_pred'])
    out = infer_sub[['periodo'] + IDS].reset_index(drop=True).copy()
    out['tn_pred'] = np.maximum(infer_sub['tn0'].to_numpy(dtype=np.float64), PARAM['clip_min'])
    out['periodo_objetivo'] = out['periodo'].map(lambda p: desplazar_meses(int(p), H))
    print(f"  [{etiqueta}] (naive) {len(out):,} filas, repitiendo tn0")
    return out


def entrenar_y_predecir_unidad(resultado, train_sub, infer_sub, etiqueta):
    if REGLA_POR_UNIDAD[resultado.get('cluster')] == 'naive':
        return predecir_naive(infer_sub, etiqueta)

    features = resultado['features']
    cat_features = resultado['cat_features']

    def params_finales(semilla):
        p = dict(resultado['hiperparametros'])
        p.update({
            'objective': resultado['objective_lgbm'], 'metric': 'mae', 'verbosity': -1,
            'boosting_type': 'gbdt', 'n_jobs': -1, 'subsample_freq': 1,
            'n_estimators': resultado['n_estimators_final'], 'seed': semilla,
            'deterministic': True, 'force_row_wise': True,
        })
        return p

    if len(train_sub) == 0:
        print(f"  [{etiqueta}] sin filas de entrenamiento, se salta")
        return pd.DataFrame(columns=IDS + ['periodo', 'periodo_objetivo', 'tn_pred'])

    X = train_sub[features]
    y = train_sub[TARGET]
    print(f"  [{etiqueta}] train {X.shape[0]:,} x {X.shape[1]}   "
          f"arboles={resultado['n_estimators_final']}")

    modelos = []
    for sem in PARAM['semillas_ensemble']:
        m = lgb.LGBMRegressor(**params_finales(sem))
        m.fit(X, y, categorical_feature=cat_features)
        modelos.append(m)
    del X, y
    gc.collect()

    if len(infer_sub) == 0:
        print(f"  [{etiqueta}] sin filas de inferencia")
        return pd.DataFrame(columns=IDS + ['periodo', 'periodo_objetivo', 'tn_pred'])

    preds = np.column_stack([m.predict(infer_sub[features]) for m in modelos])
    ctx = infer_sub[[c for c in COLS_CTX if c in infer_sub.columns]].reset_index(drop=True)
    y_pred = np.maximum(reconstruir_nivel(preds.mean(axis=1), ctx), PARAM['clip_min'])

    out = infer_sub[['periodo'] + IDS].reset_index(drop=True).copy()
    out['tn_pred'] = y_pred
    out['periodo_objetivo'] = out['periodo'].map(lambda p: desplazar_meses(int(p), H))
    return out


t0 = time.time()

if MODO == 'porCluster':
    partes = []
    for r in RESULTADOS:
        cl = r['cluster']
        train_sub = train_pd[train_pd['cluster'] == cl].reset_index(drop=True)
        infer_sub = infer_pd[infer_pd['cluster'] == cl].reset_index(drop=True)
        partes.append(entrenar_y_predecir_unidad(r, train_sub, infer_sub, f"cluster{cl}"))
    pred = pd.concat(partes, ignore_index=True)
else:
    pred = entrenar_y_predecir_unidad(RESULTADOS[0], train_pd, infer_pd, "pooled")

print(f"\nPredicciones: {len(pred):,} filas")
print(pred.groupby(['periodo', 'periodo_objetivo']).size().rename('filas').reset_index().to_string(index=False))
print(f"\ntn_pred   min {pred['tn_pred'].min():.3f}   media {pred['tn_pred'].mean():.3f}   "
      f"max {pred['tn_pred'].max():.3f}")
print(f"[{time.time()-t0:.0f}s]")


### Entrega: toneladas por producto contra la lista oficial


In [ ]:
OBJ = PARAM['periodo_objetivo'] or int(pred['periodo_objetivo'].max())
pred_obj = pred[pred['periodo_objetivo'] == OBJ]
if pred_obj.empty:
    raise RuntimeError(f"No hay predicciones para {OBJ}. Objetivos disponibles: "
                       f"{sorted(pred['periodo_objetivo'].unique())}")

por_producto = (pred_obj.groupby('product_id', as_index=False)['tn_pred']
                        .sum().rename(columns={'tn_pred': 'tn'}))
print(f"Mes objetivo {OBJ}: {len(pred_obj):,} filas -> {len(por_producto)} productos")

path_apredecir = RUTA_RAW / "product_id_apredecir201912.txt"
if not path_apredecir.exists():
    raise FileNotFoundError(f"Falta {path_apredecir}")
oficiales = pl.read_csv(path_apredecir, separator="\t").to_pandas()
print(f"Productos en la lista oficial: {len(oficiales)}")

submit = oficiales[['product_id']].merge(por_producto, on='product_id', how='left')
sin_pred = int(submit['tn'].isna().sum())
submit['tn'] = submit['tn'].fillna(0.0)
submit = submit.sort_values('product_id').reset_index(drop=True)

print(f"\nSubmit: {len(submit)} filas")
print(f"Productos SIN prediccion (van en 0): {sin_pred}")
if sin_pred > 0:
    faltantes = submit.loc[submit['tn'] == 0, 'product_id'].tolist()
    print(f"   {faltantes[:20]}{' ...' if len(faltantes) > 20 else ''}")
    if sin_pred > len(oficiales) * 0.05:
        print("   ATENCION: es mas del 5% de la lista. Revisa 01_Preprocesamiento "
              "(solo_productos_target) y el archivo_clusters si corresponde.")
print(f"\ntn   min {submit['tn'].min():.3f}   media {submit['tn'].mean():.3f}   "
      f"max {submit['tn'].max():.3f}   suma {submit['tn'].sum():,.1f}")
print(submit.head(10).to_string(index=False))

_sufijo_naive = "_conNaive" if PARAM['permitir_fallback_naive'] else ""
path_submit = DIR_EXP / f"submission_{OBJ}{_sufijo_naive}.csv"
submit.to_csv(path_submit, index=False)
print(f"\nGuardado: {path_submit}")
if not _sufijo_naive:
    print("  (permitir_fallback_naive=False -> este submit es IGUAL al de 07_Entrenamiento_final.ipynb)")

shutil.copy(path_submit, RUTA_EXP / f"submission_ultima{_sufijo_naive}.csv")
print(f"Copia    : {RUTA_EXP/f'submission_ultima{_sufijo_naive}.csv'}")


### Submit a Kaggle


In [ ]:
kaggle_dst = Path.home() / ".kaggle" / "kaggle.json"
kaggle_dst.parent.mkdir(parents=True, exist_ok=True)

if kaggle_dst.exists():
    kaggle_dst.chmod(0o600)
    print(f"Kaggle auth OK: {kaggle_dst}")
else:
    for cand in (BUCKET / "kaggle.json", BUCKET / "kaggle" / "kaggle.json"):
        if cand.exists():
            shutil.copy(cand, kaggle_dst)
            kaggle_dst.chmod(0o600)
            print(f"Kaggle auth copiada de {cand}")
            break
    else:
        print("kaggle.json NO encontrado. Bajalo de kaggle.com -> Settings -> API -> "
              f"Create New Token y dejalo en {kaggle_dst} o en {BUCKET}/kaggle.json")


def kaggle_cli(args):
    """Nunca lanza excepcion: el CSV ya esta generado y no vale la pena romper
    la corrida por el submit."""
    try:
        r = subprocess.run(['kaggle'] + args, capture_output=True, text=True)
        return r.returncode == 0, (r.stdout or '') + (r.stderr or '')
    except FileNotFoundError:
        return False, ("La CLI de kaggle no esta instalada. pip install kaggle\n"
                       "El CSV ya quedo generado; se puede subir a mano desde la web.")
    except Exception as e:
        return False, f"Error inesperado llamando a kaggle: {type(e).__name__}: {e}"


if not PARAM['submit']:
    print("PARAM['submit'] = False -> no se sube nada. El CSV ya esta generado.")
elif not kaggle_dst.exists():
    print("Sin credenciales de Kaggle: no se sube. El CSV ya esta generado.")
else:
    msg = PARAM['mensaje_submit'] or (
        f"{ELEGIDO['nombre'][:70]} | {MODO} | wape_test={ELEGIDO['wape_test']} | "
        f"naive en {_n_naive}/{len(REGLA_POR_UNIDAD)} unidad(es)")
    ok, salida = kaggle_cli(['competitions', 'submit',
                            '-c', PARAM['kaggle_competition'],
                            '-f', str(path_submit),
                            '-m', msg])
    print(f"mensaje: {msg}")
    print(salida)
    print("Submit enviado. Verificalo con la celda de abajo." if ok
          else "NO se pudo subir. El CSV esta en disco, se puede subir a mano.")

ok, salida = kaggle_cli(['competitions', 'submissions', '-c', PARAM['kaggle_competition']])
print(salida if salida.strip() else "(sin respuesta)")


In [ ]:
registro = {
    'experimento':        ELEGIDO['nombre'],
    'modo':                MODO,
    'dataset_fe':          DATASET_FE,
    'target':              TARGET,
    'periodo_objetivo':    OBJ,
    'archivo':             str(path_submit),
    'n_productos':         int(len(submit)),
    'n_sin_prediccion':    sin_pred,
    'tn_total_predicho':   float(submit['tn'].sum()),
    'semillas_ensemble':   PARAM['semillas_ensemble'],
    'muestreo_clientes':   PARAM.get('muestreo_clientes'),
    'permitir_fallback_naive': PARAM['permitir_fallback_naive'],
    'regla_por_unidad':    {str(k): v for k, v in REGLA_POR_UNIDAD.items()},
    'wape_val_en_06':      [r.get('wape_val') for r in RESULTADOS],
    'wape_naive_val_en_06': [r.get('wape_naive_val') for r in RESULTADOS],
    'wape_test_en_06':     [r.get('wape_test') for r in RESULTADOS],
    'clusters':            [r.get('cluster') for r in RESULTADOS] if MODO == 'porCluster' else None,
}
path_registro = DIR_EXP / f"submit_{OBJ}{_sufijo_naive}.json"
with open(path_registro, 'w', encoding='utf-8') as f:
    json.dump(registro, f, indent=2, ensure_ascii=False, default=str)

print(f"Registro: {path_registro}")
print(json.dumps(registro, indent=2, ensure_ascii=False, default=str))
